# DigitalTwin: local-vs-neighbourhood contrast
Compare the frozen 3-km XGBoost baseline to extra predictors:
local value minus the same-year 3-km neighbour mean.
Includes optional 1/3/5-km variants. Spatial CV is development evaluation;
no outer holdout scoring, model overwrite, or causal intervention validation.


In [ ]:
from pathlib import Path
import sys, subprocess, json
import pandas as pd
from IPython.display import display
ROOT = Path(r"C:/digitalTwin")
assert (ROOT / "scripts/23_multiscale_context.py").is_file(), "Install supplied script first."
print("Python:", sys.executable)
RUN_SMOKE = False
RUN_FULL = False


## Optional execution check
Set RUN_SMOKE=True in setup, rerun setup, then run this cell. Five trees / one fold only.

In [ ]:
if RUN_SMOKE:
    subprocess.run([sys.executable, "-u", str(ROOT / "scripts/23_multiscale_context.py"),
                    "--root", str(ROOT), "--smoke", "--confirm-only"], cwd=ROOT, check=True)
else:
    print("Smoke disabled.")


## Verify on your Windows environment
Set RUN_FULL=True in setup, rerun setup, then this cell. The fixed baseline and contrast candidate are compared on the original saved folds (10 fits).

In [ ]:
if RUN_FULL:
    subprocess.run([sys.executable, "-u", str(ROOT / "scripts/23_multiscale_context.py"),
                    "--root", str(ROOT), "--confirm-only", "--threads", "2"],
                   cwd=ROOT, check=True)
else:
    print("Training disabled; saved results can still be reviewed.")


In [ ]:
runs = []
for path in (ROOT / "models/experiments").glob("multiscale_context_*"):
    if (path / "recommendation.json").is_file():
        meta = json.loads((path / "metadata.json").read_text())
        if meta.get("fold_seed") is None:
            runs.append(path)
if not runs:
    print("No completed run on original saved folds yet.")
else:
    run = sorted(runs)[-1]
    print("Selected:", run)
    display(pd.read_csv(run / "summary.csv"))
    display(pd.read_csv(run / "year_metrics.csv"))
    print((run / "recommendation.json").read_text())


## Interpretation
Keep the baseline unless the candidate passes the comparison gate. A CV candidate
is not a deployable model; the script saves scores and predictions, not an estimator.
Reused folds and alternative partitions of the same data are not fresh validation.
Available CV-validation predictor imagery enters context, never validation targets.
Different Python/library/platform combinations can alter tree results; compare
baseline and candidate from the same run, not against scores from another machine.
